# Mushroom Classification (Edible vs Poisonous)

**Author:** Avni Singh

Predict whether a mushroom is edible or poisonous from 22 physical features (UCI Mushroom dataset).
Steps: data loading, cleaning, EDA, model comparison, evaluation, feature importance, robustness check, saving results.

> Educational project only. Never use this model to decide whether a wild mushroom is safe to eat.

## 1. Setup

In [ ]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
IMG_DIR = "images"          # plots are saved here for the README
os.makedirs(IMG_DIR, exist_ok=True)

## 2. Load the data
Download `mushrooms.csv` from Kaggle (Mushroom Classification, by UCI Machine Learning). In Colab, run the cell and select the file when asked.

In [ ]:
DATA_PATH = "mushrooms.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Select mushrooms.csv from your computer")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("Put mushrooms.csv in the same folder as this notebook")

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 3. Inspect the data
All columns are categorical letters. `class` is the target: `e` = edible, `p` = poisonous.

In [ ]:
df.info()
print("Missing values (NaN):", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df["class"].value_counts()

## 4. Data cleaning
- `stalk-root` uses `?` for unknown values. It is kept as its own category, `missing`.
- Columns with a single value carry no information and are dropped.

In [ ]:
question_marks = (df == "?").sum()
print("'?' values per column:")
print(question_marks[question_marks > 0])

constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant columns:", constant_cols)

df_clean = df.copy()
df_clean["stalk-root"] = df_clean["stalk-root"].replace("?", "missing")
df_clean = df_clean.drop(columns=constant_cols)
print("Shape after cleaning:", df_clean.shape)

## 5. Exploratory data analysis
### Class balance

In [ ]:
class_names = {"e": "Edible", "p": "Poisonous"}
colors = {"Edible": "#2a9d8f", "Poisonous": "#e76f51"}
counts = df_clean["class"].map(class_names).value_counts()

ax = counts.plot(kind="bar", color=[colors[n] for n in counts.index], rot=0,
                 figsize=(5, 4), title="Class balance")
for i, v in enumerate(counts):
    ax.text(i, v + 30, str(v), ha="center")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/class_balance.png", dpi=150)
plt.show()

### Poisonous rate by odor
Odor looks like a very strong clue. This plot shows the share of poisonous mushrooms for each odor.

In [ ]:
odor_names = {"a": "almond", "l": "anise", "c": "creosote", "y": "fishy", "f": "foul",
              "m": "musty", "n": "none", "p": "pungent", "s": "spicy"}

rate = (df_clean.groupby("odor")["class"].apply(lambda s: (s == "p").mean() * 100)
        .sort_values(ascending=False))
rate.index = [odor_names.get(c, c) for c in rate.index]

ax = rate.plot(kind="bar", color="#e76f51", figsize=(8, 4), title="Poisonous mushrooms by odor (%)")
ax.set_ylabel("% poisonous")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/poisonous_rate_by_odor.png", dpi=150)
plt.show()

### Which single feature is most informative?
For each feature, predict the majority class inside each category and measure how often that is right.

In [ ]:
single_feature_accuracy = {}
for col in df_clean.columns.drop("class"):
    majority_hits = df_clean.groupby(col)["class"].agg(lambda s: s.value_counts().iloc[0]).sum()
    single_feature_accuracy[col] = majority_hits / len(df_clean)
single_feature_accuracy = pd.Series(single_feature_accuracy).sort_values(ascending=False)

ax = single_feature_accuracy.head(10)[::-1].plot(kind="barh", figsize=(7, 5),
                                                 title="Accuracy of a single-feature rule (top 10)")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/single_feature_accuracy.png", dpi=150)
plt.show()
single_feature_accuracy.head(5)

## 6. Train/test split
80% train, 20% test, stratified so both sets keep the same edible/poisonous ratio.

In [ ]:
X = df_clean.drop(columns="class")
y = (df_clean["class"] == "p").astype(int)      # 1 = poisonous, 0 = edible

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Poisonous share  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))

## 7. Model comparison
Features are one-hot encoded inside a pipeline. Each model is scored with 5-fold cross-validation on the training set and then on the test set.

Poisonous is the positive class, so **recall** is the key metric: a *false negative* is a poisonous mushroom predicted to be edible, the dangerous mistake.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def fit_and_score(name, clf, Xtr, Xte, with_cv=True):
    pipe = make_pipeline(OneHotEncoder(handle_unknown="ignore"), clf)
    cv_acc = cross_val_score(pipe, Xtr, y_train, cv=cv, scoring="accuracy").mean() if with_cv else np.nan
    pipe.fit(Xtr, y_train)
    pred = pipe.predict(Xte)
    proba = pipe.predict_proba(Xte)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    row = {
        "Model": name,
        "CV Accuracy": round(cv_acc, 4),
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "Precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, pred), 4),
        "F1": round(f1_score(y_test, pred), 4),
        "ROC AUC": round(roc_auc_score(y_test, proba), 4),
        "False Negatives": int(fn),
        "False Positives": int(fp),
    }
    return row, pipe, pred


models = {   # simplest to most complex
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

rows, fitted, predictions = [], {}, {}
for name, clf in models.items():
    row, pipe, pred = fit_and_score(name, clf, X_train, X_test)
    rows.append(row)
    fitted[name], predictions[name] = pipe, pred

results = pd.DataFrame(rows).set_index("Model")
results

## 8. Best model and confusion matrix
The best model is the one that misses the fewest poisonous mushrooms (highest recall), then highest F1, then highest CV accuracy. If models tie, the simpler one is chosen.

In [ ]:
best = results.sort_values(["Recall", "F1", "CV Accuracy"], ascending=False).index[0]
print("Best model:", best)

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions[best], display_labels=["Edible", "Poisonous"], cmap="Blues")
plt.title(f"Confusion matrix: {best}")
plt.savefig(f"{IMG_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(classification_report(y_test, predictions[best], target_names=["Edible", "Poisonous"]))

## 9. Feature importance
One-hot columns are summed back to the original feature, so each bar is one mushroom feature.

In [ ]:
def feature_importance_by_column(pipe):
    encoder, model = pipe.steps[0][1], pipe.steps[-1][1]
    names = encoder.get_feature_names_out()
    values = model.feature_importances_ if hasattr(model, "feature_importances_") else np.abs(model.coef_[0])
    series = pd.Series(values, index=names)
    by_column = series.groupby(lambda n: n.split("_")[0]).sum()   # "odor_n" -> "odor"
    return (by_column / by_column.sum()).sort_values(ascending=False)


importance = feature_importance_by_column(fitted[best])
ax = importance.head(10)[::-1].plot(kind="barh", figsize=(7, 5), title=f"Top 10 features: {best}")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/feature_importance.png", dpi=150)
plt.show()
importance.head(5)

## 10. Robustness check: how much does the model depend on odor?
Odor alone is a very strong clue. Two extra experiments show how the model behaves **without** it and **with only** it.

In [ ]:
extra_rows = []
row, _, _ = fit_and_score("Random Forest (without odor)",
                          RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
                          X_train.drop(columns="odor"), X_test.drop(columns="odor"))
extra_rows.append(row)
row, _, _ = fit_and_score("Decision Tree (odor only)",
                          DecisionTreeClassifier(random_state=RANDOM_STATE),
                          X_train[["odor"]], X_test[["odor"]])
extra_rows.append(row)

extra_results = pd.DataFrame(extra_rows).set_index("Model")
extra_results

## 11. Save the model, results and plots

In [ ]:
os.makedirs("outputs", exist_ok=True)
joblib.dump(fitted[best], "outputs/model.pkl")
results.to_csv("outputs/results.csv")
extra_results.to_csv("outputs/results_extra.csv")
print("Saved model and results to outputs/ and plots to images/")

# In Colab: download the plots and outputs as zip files
try:
    import shutil
    from google.colab import files
    shutil.make_archive("plots", "zip", IMG_DIR)
    shutil.make_archive("outputs", "zip", "outputs")
    files.download("plots.zip")
    files.download("outputs.zip")
except ImportError:
    pass